In [24]:
import pandas as pd
import pm4py
from collections import Counter

In [25]:
df = pd.read_csv("actual_event_log.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"])

df.head()

,case_id,activity,timestamp
0,P00001,Registration,2026-10-07 20:32:41+00:00
1,P00001,Triage,2026-10-07 20:42:41+00:00
2,P00001,Lab Test,2026-10-07 20:49:41+00:00
3,P00001,Treatment,2026-10-07 21:12:41+00:00
4,P00001,Doctor Consultation,2026-10-07 21:25:41+00:00


In [26]:
df = df.sort_values(["case_id", "timestamp"])

event_log = pm4py.format_dataframe(
    df,
    case_id="case_id",
    activity_key="activity",
    timestamp_key="timestamp"
)

event_log = pm4py.convert_to_event_log(event_log)

print("Total patient cases:", len(event_log))

Total patient cases: 5000


In [27]:
net, initial_marking, final_marking = pm4py.read_pnml(
    "ideal_reference_model.pnml"
)

print("Ideal reference model loaded successfully")

Ideal reference model loaded successfully


In [28]:
fitness = pm4py.fitness_alignments(
    event_log,
    net,
    initial_marking,
    final_marking
)

print("Alignment fitness results:")
print(fitness)

aligning log, completed variants ::   0%|          | 0/18 [00:00<?, ?it/s]

Alignment fitness results:
{'percFitTraces': 20.88, 'averageFitness': 0.8848767909867811, 'percentage_of_fitting_traces': 20.88, 'average_trace_fitness': 0.8848767909867811, 'log_fitness': 0.8809347550671689}


In [29]:
token_fitness = pm4py.fitness_token_based_replay(
    event_log,
    net,
    initial_marking,
    final_marking
)

print("Token-based fitness results:")
print(token_fitness)

replaying log with TBR, completed traces ::   0%|          | 0/18 [00:00<?, ?it/s]

Token-based fitness results:
{'perc_fit_traces': 20.88, 'average_trace_fitness': 0.8531504761904762, 'log_fitness': 0.8632916244558224, 'percentage_of_fitting_traces': 20.88}


In [30]:
#Identify deviations in individual patient journeys
diagnostics = pm4py.conformance_diagnostics_alignments(
    event_log,
    net,
    initial_marking,
    final_marking
)

print("Number of patient cases analyzed:", len(diagnostics))
print("Example alignment:")
print(diagnostics[0])

aligning log, completed variants ::   0%|          | 0/18 [00:00<?, ?it/s]

Number of patient cases analyzed: 5000
Example alignment:
{'alignment': [('Registration', 'Registration'), ('Triage', 'Triage'), ('Lab Test', 'Lab Test'), ('>>', None), ('>>', None), ('Treatment', 'Treatment'), ('Doctor Consultation', 'Doctor Consultation'), ('Discharge', 'Discharge')], 'cost': 2, 'visited_states': 8, 'queued_states': 26, 'traversed_arcs': 26, 'lp_solved': 2, 'fitness': 1.0, 'bwc': 110003}


In [31]:
#Create a patient-level deviation summary
deviation_rows = []

for trace, alignment_result in zip(event_log, diagnostics):

    case_id = trace.attributes["concept:name"]
    alignment = alignment_result.get("alignment", [])

    log_moves = []
    model_moves = []

    for log_move, model_move in alignment:

        # Activity exists in actual log but not in reference model
        if model_move == ">>":
            log_moves.append(log_move)

        # Activity expected by model but missing from actual log
        elif log_move == ">>":
            model_moves.append(model_move)

    deviation_rows.append({
        "case_id": case_id,
        "log_deviation_count": len(log_moves),
        "model_deviation_count": len(model_moves),
        "total_deviations": len(log_moves) + len(model_moves),
        "log_deviations": ", ".join(map(str, log_moves)),
        "missing_model_activities": ", ".join(map(str, model_moves))
    })

deviation_df = pd.DataFrame(deviation_rows)

deviation_df.head(10)

,case_id,log_deviation_count,model_deviation_count,total_deviations,log_deviations,missing_model_activities
0,P00001,0,2,2,,"None, None"
1,P00002,1,3,4,X-Ray,"None, None, Treatment"
2,P00003,0,3,3,,"None, None, Treatment"
3,P00004,2,2,4,"Return to Triage, Triage","None, None"
4,P00005,0,2,2,,"None, None"
5,P00006,0,2,2,,"None, None"
6,P00007,0,3,3,,"None, None, Treatment"
7,P00008,0,4,4,,"None, None, None, Treatment"
8,P00009,0,2,2,,"None, None"
9,P00010,0,3,3,,"None, None, Treatment"


In [32]:
#Find the most common deviations
deviation_summary = (
    deviation_df.groupby(
        ["log_deviations", "missing_model_activities"],
        dropna=False
    )
    .size()
    .reset_index(name="patient_count")
    .sort_values("patient_count", ascending=False)
)

deviation_summary.head(20)

,log_deviations,missing_model_activities,patient_count
2,,"None, None, Treatment",1453
1,,"None, None, None, Treatment",1121
0,,"None, None",1044
4,"Return to Triage, Triage","None, None, Treatment",625
3,"Return to Triage, Triage","None, None",345
6,X-Ray,"None, None, Treatment",241
5,"Triage, X-Ray, Return to Triage","None, None, Treatment",171


In [33]:
#calculate how many patient cases have at least one deviation:
total_cases = len(deviation_df)

deviated_cases = (
    deviation_df["total_deviations"] > 0
).sum()

compliant_cases = total_cases - deviated_cases

print("Total patient cases:", total_cases)
print("Cases with deviations:", deviated_cases)
print("Cases without detected deviations:", compliant_cases)

print(
    "Deviation rate:",
    round(deviated_cases / total_cases * 100, 2),
    "%"
)

Total patient cases: 5000
Cases with deviations: 5000
Cases without detected deviations: 0
Deviation rate: 100.0 %


In [34]:
#Save your results
deviation_df.to_csv(
    "patient_level_deviations.csv",
    index=False
)

deviation_summary.to_csv(
    "deviation_summary.csv",
    index=False
)

print("Part 4 results saved successfully!")

Part 4 results saved successfully!


In [35]:
print("First patient alignment:")
print(diagnostics[0]["alignment"])

First patient alignment:
[('Registration', 'Registration'), ('Triage', 'Triage'), ('Lab Test', 'Lab Test'), ('>>', None), ('>>', None), ('Treatment', 'Treatment'), ('Doctor Consultation', 'Doctor Consultation'), ('Discharge', 'Discharge')]


In [36]:
print("First patient case:")
print(event_log[0].attributes["concept:name"])

print("\nActual activities:")
print([event["concept:name"] for event in event_log[0]])

First patient case:
P00001

Actual activities:
['Registration', 'Triage', 'Lab Test', 'Treatment', 'Doctor Consultation', 'Discharge']


In [37]:
print("Alignment result keys:")
print(diagnostics[0].keys())

Alignment result keys:
dict_keys(['alignment', 'cost', 'visited_states', 'queued_states', 'traversed_arcs', 'lp_solved', 'fitness', 'bwc'])


In [38]:
deviation_rows = []

for trace, alignment_result in zip(event_log, diagnostics):

    case_id = trace.attributes["concept:name"]
    alignment = alignment_result["alignment"]

    log_moves = []
    model_moves = []

    for log_move, model_move in alignment:

        # Actual activity not matched by the reference model
        if log_move != ">>" and model_move == ">>":
            log_moves.append(log_move)

        # Expected model activity missing from actual journey
        elif log_move == ">>" and model_move is not None:
            model_moves.append(model_move)

    deviation_rows.append({
        "case_id": case_id,
        "log_deviation_count": len(log_moves),
        "model_deviation_count": len(model_moves),
        "total_deviations": len(log_moves) + len(model_moves),
        "log_deviations": ", ".join(map(str, log_moves)),
        "missing_model_activities": ", ".join(map(str, model_moves))
    })

deviation_df = pd.DataFrame(deviation_rows)

deviation_df.head(10)

,case_id,log_deviation_count,model_deviation_count,total_deviations,log_deviations,missing_model_activities
0,P00001,0,0,0,,
1,P00002,1,1,2,X-Ray,Treatment
2,P00003,0,1,1,,Treatment
3,P00004,2,0,2,"Return to Triage, Triage",
4,P00005,0,0,0,,
5,P00006,0,0,0,,
6,P00007,0,1,1,,Treatment
7,P00008,0,1,1,,Treatment
8,P00009,0,0,0,,
9,P00010,0,1,1,,Treatment


In [39]:
total_cases = len(deviation_df)

deviated_cases = (
    deviation_df["total_deviations"] > 0
).sum()

compliant_cases = total_cases - deviated_cases

print("Total patient cases:", total_cases)
print("Cases with deviations:", deviated_cases)
print("Cases without detected deviations:", compliant_cases)

print(
    "Deviation rate:",
    round(deviated_cases / total_cases * 100, 2),
    "%"
)

Total patient cases: 5000
Cases with deviations: 3956
Cases without detected deviations: 1044
Deviation rate: 79.12 %


In [40]:
deviation_summary = (
    deviation_df.groupby(
        ["log_deviations", "missing_model_activities"],
        dropna=False
    )
    .size()
    .reset_index(name="patient_count")
    .sort_values("patient_count", ascending=False)
)

deviation_summary.head(10)

,log_deviations,missing_model_activities,patient_count
1,,Treatment,2574
0,,,1044
3,"Return to Triage, Triage",Treatment,625
2,"Return to Triage, Triage",,345
5,X-Ray,Treatment,241
4,"Triage, X-Ray, Return to Triage",Treatment,171


In [41]:
deviation_df.to_csv(
    "patient_level_deviations.csv",
    index=False
)

deviation_summary.to_csv(
    "deviation_summary.csv",
    index=False
)

print("Corrected Part 4 results saved!")

Corrected Part 4 results saved!


In [42]:
print("CONFORMANCE CHECKING RESULTS")
print("----------------------------")

print("Total patient cases:", total_cases)
print("Cases with detected deviations:", deviated_cases)
print("Cases without detected deviations:", compliant_cases)

print(
    "Deviation rate:",
    round(deviated_cases / total_cases * 100, 2),
    "%"
)

print("\nAlignment fitness:")
print(fitness)

print("\nToken-based replay fitness:")
print(token_fitness)

CONFORMANCE CHECKING RESULTS
----------------------------
Total patient cases: 5000
Cases with detected deviations: 3956
Cases without detected deviations: 1044
Deviation rate: 79.12 %

Alignment fitness:
{'percFitTraces': 20.88, 'averageFitness': 0.8848767909867811, 'percentage_of_fitting_traces': 20.88, 'average_trace_fitness': 0.8848767909867811, 'log_fitness': 0.8809347550671689}

Token-based replay fitness:
{'perc_fit_traces': 20.88, 'average_trace_fitness': 0.8531504761904762, 'log_fitness': 0.8632916244558224, 'percentage_of_fitting_traces': 20.88}
